# EDA: Fantasy Points Projections

Initial exploratory pass over the modeling dataset (`data/cached_features.parquet`) before any feature engineering or modeling decisions.

Goals:
- Understand the shape, coverage, and quality of the data
- Characterize the dependent variable (`fantasy_points`) — distribution, by position, over time
- Surface missingness, outliers, and obvious data issues
- Identify which features correlate most with the target, and where multicollinearity lives
- Note anything that should inform modeling choices (transforms, position-specific models, stratification)

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

DATA_PATH = "../data/cached_features.parquet"
TARGET = "fantasy_points"

## 1. Load data and get oriented

In [ ]:
df = pd.read_parquet(DATA_PATH)
print(f"shape: {df.shape}")
df.head()

In [ ]:
df.dtypes.value_counts()

In [ ]:
print(f"Seasons: {df['season'].min()}–{df['season'].max()}")
print(f"Weeks per season: {sorted(df['week'].unique())}")
print(f"Positions: {df['position'].value_counts().to_dict()}")
print(f"Unique players: {df['player_id'].nunique()}")
print(f"Unique teams: {df['team'].nunique()}")

In [ ]:
rows_per_season = df.groupby("season").size()
fig, ax = plt.subplots(figsize=(10, 4))
rows_per_season.plot(kind="bar", ax=ax, color="#4C72B0")
ax.set_title("Player-week rows per season")
ax.set_xlabel("Season")
ax.set_ylabel("Rows")
plt.tight_layout()
plt.show()

## 2. Missing data overview

Focus on the raw stat columns and target first, then a broader pass across all 279 columns to flag anything mostly-empty.

In [ ]:
core_cols = [
    "fantasy_points", "passing_yards", "rushing_yards", "receiving_yards",
    "targets", "receptions", "snap_count", "snap_share", "target_share",
    "air_yards", "utilization_score",
]
df[core_cols].isna().mean().sort_values(ascending=False).to_frame("pct_missing")

In [ ]:
missing_pct = df.isna().mean().sort_values(ascending=False)
print(f"Columns >50% missing: {(missing_pct > 0.5).sum()} / {len(missing_pct)}")
missing_pct[missing_pct > 0.5].head(30).to_frame("pct_missing")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(missing_pct, bins=40, ax=ax, color="#DD8452")
ax.set_title("Distribution of missingness across all columns")
ax.set_xlabel("Fraction missing")
plt.tight_layout()
plt.show()

## 3. Dependent variable: `fantasy_points`

In [ ]:
target = df[TARGET].dropna()
print(f"n = {len(target)} (dropped {df[TARGET].isna().sum()} NaN)")
target.describe()

In [ ]:
print(f"skew:    {stats.skew(target):.3f}")
print(f"kurtosis: {stats.kurtosis(target):.3f}")
print(f"zeros:   {(target == 0).mean():.1%}")
print(f"negative: {(target < 0).mean():.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(target, bins=60, kde=True, ax=axes[0], color="#4C72B0")
axes[0].axvline(target.mean(), color="red", ls="--", label=f"mean={target.mean():.1f}")
axes[0].axvline(target.median(), color="green", ls="--", label=f"median={target.median():.1f}")
axes[0].set_title("Distribution of fantasy_points")
axes[0].legend()

stats.probplot(target, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q plot vs. normal")

plt.tight_layout()
plt.show()

### By position

Fantasy scoring differs structurally by position (QBs score differently than RB/WR/TE), so pooling positions is likely wrong for anything beyond a first look.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df, x="position", y=TARGET, order=["QB", "RB", "WR", "TE"], ax=axes[0])
axes[0].set_title("fantasy_points by position")

for pos in ["QB", "RB", "WR", "TE"]:
    sub = df.loc[df["position"] == pos, TARGET].dropna()
    sns.kdeplot(sub, ax=axes[1], label=pos, fill=True, alpha=0.15)
axes[1].set_title("fantasy_points density by position")
axes[1].legend()

plt.tight_layout()
plt.show()

df.groupby("position")[TARGET].describe()

### Over time

Checking for scoring drift across seasons (rule changes, offensive environment shifts) and within-season patterns (bye weeks, playoff usage change).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

df.groupby("season")[TARGET].mean().plot(marker="o", ax=axes[0])
axes[0].set_title("Mean fantasy_points by season")
axes[0].set_ylabel("mean fantasy_points")

df.groupby("week")[TARGET].mean().plot(marker="o", ax=axes[1], color="#55A868")
axes[1].set_title("Mean fantasy_points by week number")
axes[1].set_ylabel("mean fantasy_points")

plt.tight_layout()
plt.show()

### Outliers

Top and bottom performances — sanity-check these are real games, not data errors.

In [ ]:
id_cols = ["name", "position", "team", "opponent", "season", "week", TARGET]
print("Top 10 performances:")
display(df.nlargest(10, TARGET)[id_cols])
print("\nBottom 10 performances:")
display(df.nsmallest(10, TARGET)[id_cols])

In [ ]:
q1, q3 = target.quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
n_outliers = ((target < lo) | (target > hi)).sum()
print(f"IQR fence: [{lo:.1f}, {hi:.1f}]")
print(f"Outliers by 1.5*IQR rule: {n_outliers} ({n_outliers/len(target):.1%})")

## 4. Relationship between target and key features

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=[TARGET, "id"], errors="ignore")
corrs = numeric_df.corrwith(df[TARGET]).dropna().sort_values(key=np.abs, ascending=False)

top_n = 25
fig, ax = plt.subplots(figsize=(8, 9))
top_corrs = corrs.head(top_n)
colors = ["#4C72B0" if v > 0 else "#C44E52" for v in top_corrs]
top_corrs.iloc[::-1].plot(kind="barh", ax=ax, color=colors[::-1])
ax.set_title(f"Top {top_n} features correlated with fantasy_points")
ax.set_xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

In [ ]:
top_feats = corrs.head(12).index.tolist()
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
for ax, feat in zip(axes.flat, top_feats):
    sample = df[[feat, TARGET]].dropna().sample(min(5000, df[feat].notna().sum()), random_state=0)
    ax.scatter(sample[feat], sample[TARGET], s=4, alpha=0.25, color="#4C72B0")
    ax.set_title(f"{feat} (r={corrs[feat]:.2f})", fontsize=10)
plt.tight_layout()
plt.show()

## 5. Multicollinearity among top predictors

Cheap check before feature selection / regularization decisions.

In [ ]:
top_feats_20 = corrs.head(20).index.tolist()
corr_matrix = df[top_feats_20 + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr_matrix, cmap="coolwarm", center=0, annot=False, ax=ax, square=True)
ax.set_title("Correlation matrix: top predictors + target")
plt.tight_layout()
plt.show()

## 6. Categorical / structural fields

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

df["position"].value_counts().plot(kind="bar", ax=axes[0], color="#4C72B0")
axes[0].set_title("Row count by position")

df["team"].value_counts().plot(kind="bar", ax=axes[1], color="#55A868", figsize=(14, 4.5))
axes[1].set_title("Row count by team")
axes[1].tick_params(axis="x", labelsize=7)

plt.tight_layout()
plt.show()

In [ ]:
playoff_cols = ["is_playoff_week", "is_wild_card", "is_divisional", "is_conference_championship", "is_super_bowl"]
df[playoff_cols].sum().to_frame("row_count")

## 7. Key takeaways

*(fill in after running — placeholders below reflect what to check for)*

- **Distribution shape**: is `fantasy_points` right-skewed with a floor near 0? Does a log1p or sqrt transform normalize it, or is a position-specific model more appropriate than a transform?
- **Position pooling**: do QB/RB/WR/TE distributions differ enough that a single global model is the wrong framing (consistent with this repo's per-position component models)?
- **Missingness**: which engineered features (rolling/lag) are missing early in a player's career or season — confirm the `_imputed`/`_missing` flag columns already in the dataset handle this correctly.
- **Leakage risk**: features like `projection_1w`, `projection_4w`, `predicted_points`, `fp_season_avg` are themselves model outputs or same-week aggregates — flag for exclusion from any "predict this week from prior weeks" feature set.
- **Outliers**: are top/bottom performances legitimate (e.g., a 50-point game) or data errors worth filtering/winsorizing?
- **Top correlated features**: sanity-check they match domain intuition (targets, red zone usage, snap share) rather than pointing at leakage.